In [1]:
from google.colab import drive

# Google Drive mount karo
drive.mount('/content/drive')

print(" Drive mounted successfully!")

Mounted at /content/drive
 Drive mounted successfully!


In [2]:
import shutil

# Drive se dataset ka path
drive_dataset_path = "/content/drive/MyDrive/Pneumonia_90_10_Splitted"

# Colab local storage path
local_dataset_path = "/content/Pneumonia_90_10_Splitted"

# Agar already local me hai to pehle delete kar do (fresh copy ke liye)
shutil.rmtree(local_dataset_path, ignore_errors=True)

# Copy from Drive to local
shutil.copytree(drive_dataset_path, local_dataset_path)

print(" Dataset copied to local Colab storage:", local_dataset_path)


 Dataset copied to local Colab storage: /content/Pneumonia_90_10_Splitted


In [5]:
# ============================================
# Pneumonia Stage-2 training: EfficientNetB4
# Classes: Bacterial Pneumonia vs Viral Pneumonia
# Target: 95% accuracy
# ============================================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import EfficientNetB4, preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from google.colab import drive

# =========================
# Step 1: Mount Google Drive
# =========================
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/pneumonia_detection_system"
MODEL_DIR = os.path.join(BASE_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "efficientnetb4_stage2_bact_vs_viral.keras")
DATASET_DIR = "/content/Pneumonia_90_10_Splitted/train"

# =========================
# Step 2: Data Preprocessing (ONLY training, no split)
# =========================
IMG_SIZE = (380, 380)   # EfficientNetB4 default input size
BATCH_SIZE = 16         # B4 is heavier, reduce batch size

classes_to_use = ['Bacterial Pneumonia', 'Viral Pneumonia']

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest"
)

train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    classes=classes_to_use
)

print("Class indices:", train_generator.class_indices)

# Class Weights
labels = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights_array))
print("Class Weights:", class_weights)

# =========================
# Step 3: Build EfficientNetB4 Model
# =========================
base_model = EfficientNetB4(weights="imagenet", include_top=False, input_shape=(380, 380, 3))
base_model.trainable = False  # Freeze initially

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# =========================
# Step 4: Training Phase 1 (Frozen Base) - 20 epochs
# =========================
callbacks = [
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='loss', mode='min', verbose=1),
    EarlyStopping(monitor='loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='loss', factor=0.3, patience=4, min_lr=1e-6, verbose=1)
]

history1 = model.fit(
    train_generator,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weights
)

# =========================
# Step 5: Fine-Tuning Phase 2 (Unfreeze last 100 layers) - 40 epochs
# =========================
base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history2 = model.fit(
    train_generator,
    epochs=40,
    callbacks=callbacks,
    class_weight=class_weights
)

# =========================
# Step 6: Save Final Model
# =========================
model.save(MODEL_PATH)
print(f"Model saved at {MODEL_PATH}")

# =========================
# Step 7: Evaluate on Test Set
# =========================
TEST_DIR = "/content/Pneumonia_90_10_Splitted/test"
if os.path.exists(TEST_DIR):
    test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    test_generator = test_datagen.flow_from_directory(
        TEST_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False,
        classes=classes_to_use
    )

    loss, acc = model.evaluate(test_generator, verbose=1)
    print("Test loss:", loss, "Test accuracy:", acc)

    from sklearn.metrics import confusion_matrix, classification_report
    y_true = test_generator.classes
    y_prob = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=list(test_generator.class_indices.keys())))
else:
    print("Test folder not found at:", TEST_DIR)

# =========================
# Step 8: Print Last Conv Layer (For Grad-CAM)
# =========================
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        print("Last Conv Layer for Grad-CAM:", layer.name)
        break


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 9000 images belonging to 2 classes.
Class indices: {'Bacterial Pneumonia': 0, 'Viral Pneumonia': 1}
Class Weights: {0: np.float64(1.0), 1: np.float64(1.0)}


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 380, 380,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_4         │ (None, 380, 380,  │          0 │ input_layer_2[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_2     │ (None, 380, 380,  │          7 │ rescaling_4[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_5         │ (None, 380, 380,  │          0 │ normalization_2[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 381, 381,  │          0 │ rescaling_5[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 190, 190,  │      1,296 │ stem_conv_pad[0]… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 190, 190,  │        192 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 190, 190,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 190, 190,  │        432 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 190, 190,  │        192 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 190, 190,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 48)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 48)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 12)  │        588 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 48)  │        624 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 190, 190,  │          0 │ block1a_activati… │
│ (Multiply)          │ 48)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 190, 190,  │      1,152 │ block1a_se_excit

 Total params: 18,592,865 (70.93 MB)

 Trainable params: 919,042 (3.51 MB)

 Non-trainable params: 17,673,823 (67.42 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 690ms/step - accuracy: 0.6651 - loss: 0.6187
Epoch 1: loss improved from inf to 0.58664, saving model to /content/drive/MyDrive/pneumonia_detection_system/model/efficientnetb4_stage2_bact_vs_viral.keras
563/563 ━━━━━━━━━━━━━━━━━━━━ 437s 694ms/step - accuracy: 0.6651 - loss: 0.6186 - learning_rate: 1.0000e-04
Epoch 2/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 626ms/step - accuracy: 0.7276 - loss: 0.5497
Epoch 2: loss improved from 0.58664 to 0.54321, saving model to /content/drive/MyDrive/pneumonia_detection_system/model/efficientnetb4_stage2_bact_vs_viral.keras
563/563 ━━━━━━━━━━━━━━━━━━━━ 380s 629ms/step - accuracy: 0.7276 - loss: 0.5497 - learning_rate: 1.0000e-04
Epoch 3/20
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 634ms/step - accuracy: 0.7474 - loss: 0.5263
Epoch 3: loss improved from 0.54321 to 0.53300, saving model to /content/drive/MyDrive/pneumonia_detection_system/model/efficientnetb4_stage2_bact_vs_viral.keras
563/563 ━━━━━━━━━━━━━━━━━━━━ 359s 637ms/

In [6]:
# =========================
# Step 8: Print Last Conv Layer (For Grad-CAM)
# =========================
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        print("Last Conv Layer for Grad-CAM:", layer.name)
        break

Last Conv Layer for Grad-CAM: top_conv
